# GPT-style Large Language Model (LLM) Training

This notebook implements a Decoder-only Transformer (GPT architecture) from scratch to perform next-token prediction on '../data/the-verdict.txt'.

In [1]:
import urllib.request
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
import tiktoken
from tqdm import tqdm

# Optimized Hyperparameters for a 4GB GPU
batch_size = 16       # Increased from 4 for better GPU utilization
block_size = 256      # Context length
n_embd = 384          # Increased from 128 for more expressive representations
n_head = 6            # Increased from 4 (384 / 6 = 64 head dimension)
n_layer = 6           # Increased from 4 for a deeper network
dropout = 0.2         # Slightly higher dropout for regularization on a larger model
learning_rate = 3e-4  # Standard GPT-3 style learning rate for AdamW



if torch.cuda.is_available():
    device = 'cuda' 
    # Speed up matrix multiplications on NVIDIA hardware with Tensor Cores
    torch.set_float32_matmul_precision('high')
else:
    device = 'cpu'

eval_iters = 10
eval_interval = 100
max_iters = 500

print(f"Using device: {device}")


c:\Users\61481\Softwares\large-language-model\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:295: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


Using device: cuda


## Data Preparation
Download the dataset, tokenize it using `tiktoken` (`cl100k_base`), and split it into training (90%) and validation (10%) sets.

In [2]:
# Download dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
filepath = "tinyshakespeare.txt"
try:
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()
except FileNotFoundError:
    print(f"Downloading {filepath}...")
    urllib.request.urlretrieve(url, filepath)
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

print(f"Dataset length in characters: {len(text)}")

# Tokenize
enc = tiktoken.get_encoding("cl100k_base")
tokens = enc.encode(text)
print(f"Dataset length in tokens: {len(tokens)}")
vocab_size = enc.n_vocab
print(f"Vocabulary size: {vocab_size}")

# Split into train and val (90/10)
n = int(0.9 * len(tokens))
train_data = torch.tensor(tokens[:n], dtype=torch.long)
val_data = torch.tensor(tokens[n:], dtype=torch.long)


Dataset length in characters: 1115393
Dataset length in tokens: 301829
Vocabulary size: 100277


## Dataset Class
Implement `GPTDataset` to return sliding window chunks of `block_size` tokens for `x` and shifted tokens for `y`.

In [3]:
class GPTDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        # Get chunk of size block_size + 1
        chunk = self.data[idx:idx + self.block_size + 1]
        x = chunk[:-1] # Inputs
        y = chunk[1:]  # Targets
        return x, y

train_dataset = GPTDataset(train_data, block_size)
val_dataset = GPTDataset(val_data, block_size)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)


## Model Architecture
Building the Transformer components: Multi-Head Attention, FeedForward Network, and the Transformer Block.

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0, "Embedding dimension must be divisible by number of heads"
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        
        # Key, query, value projections for all heads
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        # Output projection
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)
        
        # Regularization
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        
        # Causal mask to ensure attention is only applied to past tokens
        self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.size() # Batch size, Sequence length, Embedding dim

        # Calculate query, key, values for all heads
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)
        
        # Reshape to (B, n_head, T, head_dim)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # Causal self-attention: (B, n_head, T, head_dim) x (B, n_head, head_dim, T) -> (B, n_head, T, T)
        att = (q @ k.transpose(-2, -1)) * (1.0 / (self.head_dim ** 0.5))
        
        # Apply causal mask
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        
        # Weighted sum of values: (B, n_head, T, T) x (B, n_head, T, head_dim) -> (B, n_head, T, head_dim)
        y = att @ v
        
        # Re-assemble all head outputs side by side
        y = y.transpose(1, 2).contiguous().view(B, T, C)

        # Output projection
        y = self.resid_dropout(self.c_proj(y))
        return y


In [5]:
class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


In [6]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = MultiHeadAttention(n_embd, n_head, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd, dropout)

    def forward(self, x):
        # Pre-LayerNorm formulation with residual connections
        x = x + self.attn(self.ln_1(x))
        x = x + self.ffwd(self.ln_2(x))
        return x


In [7]:
class GPTModel(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        
        # Token and Position embeddings
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        
        # Transformer Blocks
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, dropout) for _ in range(n_layer)])
        
        # Final LayerNorm
        self.ln_f = nn.LayerNorm(n_embd)
        
        # Language Model Head
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        
        # Weight initialization
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        
        # Ensure sequence length does not exceed block_size
        idx = idx[:, -self.block_size:]
        B, T = idx.shape
        
        # Get embeddings
        tok_emb = self.token_embedding_table(idx) # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device)) # (T, n_embd)
        x = tok_emb + pos_emb # (B, T, n_embd)
        
        # Pass through blocks and final layernorm
        x = self.blocks(x) # (B, T, n_embd)
        x = self.ln_f(x) # (B, T, n_embd)
        
        # Compute logits
        logits = self.lm_head(x) # (B, T, vocab_size)
        
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits_reshaped = logits.view(B * T, C)
            targets_reshaped = targets.view(B * T)
            loss = F.cross_entropy(logits_reshaped, targets_reshaped)
            
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # Crop idx to the last block_size tokens
            idx_cond = idx[:, -self.block_size:]
            # Get the predictions
            logits, _ = self(idx_cond)
            # Focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # Apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # Append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


## Training Setup and Loop
Initialize the model and AdamW optimizer. Define an evaluation function, and run the training loop.

In [8]:
# Initialize model and optimizer
model = GPTModel(vocab_size, n_embd, n_head, n_layer, block_size, dropout)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split, loader in [('train', train_loader), ('val', val_loader)]:
        losses = torch.zeros(eval_iters)
        # Manually grab batches to estimate loss
        iterator = iter(loader)
        for k in range(eval_iters):
            try:
                X, Y = next(iterator)
            except StopIteration:
                iterator = iter(loader)
                X, Y = next(iterator)
            X, Y = X.to(device), Y.to(device)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# Training Loop
iterator = iter(train_loader)
for iter_num in tqdm(range(max_iters), desc="Training"):
    # Evaluate the loss periodically
    if iter_num % eval_interval == 0 or iter_num == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter_num}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # Sample a batch of data
    try:
        xb, yb = next(iterator)
    except StopIteration:
        iterator = iter(train_loader)
        xb, yb = next(iterator)
        
    xb, yb = xb.to(device), yb.to(device)

    # Forward pass
    logits, loss = model(xb, yb)
    
    # Backward pass and optimize
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


Training:   0%|          | 0/500 [00:00<?, ?it/s]

step 0: train loss 11.6226, val loss 11.5833


Training:  20%|██        | 101/500 [15:50<1:50:37, 16.64s/it]

step 100: train loss 6.5694, val loss 6.7256


Training:  40%|████      | 201/500 [31:15<1:21:28, 16.35s/it]

...

step 300: train loss 4.9528, val loss 5.4437


Training:  80%|████████  | 401/500 [1:04:06<27:58, 16.96s/it]

step 400: train loss 4.6315, val loss 5.2511


Training: 100%|██████████| 500/500 [1:20:25<00:00,  9.65s/it]

step 499: train loss 4.4677, val loss 5.1520


## Inference
Use the trained model to generate a sequence of tokens.

In [10]:
# Generate text from the trained model
context_text = "Hello "
context_tokens = enc.encode(context_text)
context_tensor = torch.tensor([context_tokens], dtype=torch.long, device=device)

model.eval()
print("Generating text...")
generated_tokens = model.generate(context_tensor, max_new_tokens=200)
generated_text = enc.decode(generated_tokens[0].tolist())

print("\n--- Generated Text ---")
print(generated_text)


Generating text...

--- Generated Text ---
Hello  south, child nor it assisting PSHow would seize their own cited:
Asicoer will;
Your friends,
That in your am I never pl tinderful adm.IsDBNull some proud,
Even in the virgin turning,
Like valiant heavens thyself in a better son
To bed;
Where where arise home, when spiltum, but every Clare.

CLAUDIO

First Murdererzie thence to his honour hol mind:
Be shortly shall speak.

Farewell.

Lords:
Not in therish him
Again pronounce this marriage, I am nay, but diest thee better than but starON:
Have.aves on in what he?
 warrior;
Or what I have made madam! the deputy wash'd in this doubt not come to courthouse queen down.
3 KING HENRY BOLING RICHARD II:
Thou art thou post.

Lordge, Edward's rich that sad?

WARWICK:
But quarrel of limbs is no more than thy masters,
